# NuSmoothie + Fisher information

Simulate one event with the three learned surrogates (`NuSmoothie`), look at it, then
compute the Fisher information / angular resolution for the same event and geometry
with `FlowFisherResolutionLoss`.

Everything tunable lives in the config cell; the rest just runs.

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import numpy as np
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import importlib
import nugget
from nugget.surrogates.NuSmoothie import NuSmoothie
from nugget.losses.fisher_info_flow import FlowFisherResolutionLoss

## Config

In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

HIT_CKPT   = './flow_models/best_mc_hit_model_v3.pt'
LY_CKPT    = './flow_models/best_mc_ly_muon_flow_model_v1.pt'
ATIME_CKPT = './flow_models/best_mc_atime_muon_flow_model_v3.pt'

# --- geometry ---
N_STRINGS        = 19
POINTS_PER_STRING = 12
STRING_SPACING   = 50.0     # m between strings
Z_SPACING        = 40.0     # m between OMs on a string
DOMAIN           = 1200.0

# --- events ---
N_EVENTS   = 4
E_MIN, E_MAX = 1e5, 1e6
CYL_RADIUS, CYL_HEIGHT = 150.0, 250.0
EVENT_SEED = 0

# --- surrogate / Fisher ---
N_ODE_STEPS = 16        # NuSmoothie sampling
LY_N_SAMPLES = 8        # MC samples for E[q | hit]
FISHER_N_STEPS = 4      # ODE steps inside the Fisher (memory is linear in this)
FISHER_N_QUAD  = 12     # quadrature nodes per likelihood
FISHER_CHUNK   = 512    # rows per backward; ~2 MB/row at n_steps=8 in float64

## Geometry

In [ ]:
geometry = nugget.geometries.DynamicString.DynamicString(
    device=device,
    hex_type='hexagonal',
    domain_size=DOMAIN,
    dim=3,
    n_strings=N_STRINGS,
    points_per_string=POINTS_PER_STRING,
    custom_string_spacing=STRING_SPACING,
    custom_z_spacing=Z_SPACING,
)
geom_dict = geometry.initialize_points()

pts = geom_dict['points_3d'].detach().cpu().numpy()
sxy = geom_dict['string_xy'].detach().cpu().numpy()
print(f"{len(pts)} OMs on {len(sxy)} strings")
print(f"  x {pts[:,0].min():7.1f} .. {pts[:,0].max():7.1f} m")
print(f"  y {pts[:,1].min():7.1f} .. {pts[:,1].max():7.1f} m")
print(f"  z {pts[:,2].min():7.1f} .. {pts[:,2].max():7.1f} m")

## Events

`CylinderSampler` returns one dict per event with `energy`, `zenith`, `azimuth`,
`position` and `direction` — exactly the keys both `NuSmoothie` and the Fisher loss
expect, so no conversion is needed.

In [ ]:
sampler = nugget.samplers.cyl_sampler.CylinderSampler(
    device=device,
    event_type='signal',
    domain_size=DOMAIN,
    E_min=E_MIN, E_max=E_MAX,
    energy_dist='log_uniform',
    uniform_zenith_sampling=True,
    random_position_along_ray=True,
    find_exact_intersection=False,
    cylinder_radius=CYL_RADIUS,
    cylinder_height=CYL_HEIGHT,
    cylinder_center=[0, 0, 0],
)

torch.manual_seed(EVENT_SEED)
events = sampler.sample_events(N_EVENTS)

def track_frame(ev, P):
    """(d_perp, d_long) of every OM w.r.t. this event's track."""
    z, a = float(ev['zenith']), float(ev['azimuth'])
    u = np.array([np.sin(z)*np.cos(a), np.sin(z)*np.sin(a), np.cos(z)])
    rel = P - np.asarray(ev['position'], dtype=float).reshape(1, 3)
    dl = rel @ u
    return np.linalg.norm(rel - dl[:, None] * u, axis=1), dl, u

print(f"{'ev':>3} {'E [GeV]':>10} {'zenith':>7} {'azimuth':>8} {'min d_perp [m]':>15}")
for i, ev in enumerate(events):
    dp, dl, _ = track_frame(ev, pts)
    print(f"{i:>3} {float(ev['energy']):>10.3g} {float(ev['zenith']):>7.3f} "
          f"{float(ev['azimuth']):>8.3f} {dp.min():>15.1f}")

## The combined surrogate

In [ ]:
nusmoothie = NuSmoothie(
    device=device,
    domain_size=8000,          # overwritten per model by load_model
    hit_checkpoint=HIT_CKPT,
    ly_checkpoint=LY_CKPT,
    atime_checkpoint=ATIME_CKPT,
    n_steps=N_ODE_STEPS,
    ly_n_samples=LY_N_SAMPLES,
)

## Simulate one event

`light_yield_surrogate` returns the expected photons `pi(c) . E[q | c, q>=1]` per
optical module. With the default `pmt_mode='sum'` each OM pools its 16 PMTs (the
template is read from the geometry the models were trained on); `pmt_mode='split'`
would give one entry per PMT instead.

In [ ]:
EVENT = 0
ev = events[EVENT]

light_yield = nusmoothie.light_yield_surrogate(
    event_params=ev, opt_point=geom_dict['points_3d'])
ly = light_yield.detach().cpu().numpy()

dp, dl, u = track_frame(ev, pts)
print(f"event {EVENT}: E = {float(ev['energy']):.3g} GeV, "
      f"zenith = {float(ev['zenith']):.3f}, closest OM = {dp.min():.1f} m")
print(f"expected photons: total {ly.sum():.2f}, max {ly.max():.3f}, "
      f"OMs above 0.01 = {(ly > 0.01).sum()}")

## What it looks like

In [ ]:
fig = plt.figure(figsize=(13, 5.5))

# --- 3-D view, OMs coloured by expected light ---
ax = fig.add_subplot(121, projection='3d')
dark = ly <= 1e-3
ax.scatter(pts[dark, 0], pts[dark, 1], pts[dark, 2], s=4, c='0.8', alpha=.5)
if (~dark).any():
    sc = ax.scatter(pts[~dark, 0], pts[~dark, 1], pts[~dark, 2],
                    s=18 + 120 * ly[~dark] / max(ly.max(), 1e-9),
                    c=ly[~dark], cmap='plasma', norm=plt.matplotlib.colors.LogNorm())
    fig.colorbar(sc, ax=ax, shrink=.6, label='expected photons / OM')

v = np.asarray(ev['position'], dtype=float).reshape(3)
travel = -u if getattr(nusmoothie.hit_model, 'track_dir_is_arrival', False) else u
s_line = np.linspace(-400, 800, 2)
line = v[None, :] + s_line[:, None] * travel[None, :]
ax.plot(line[:, 0], line[:, 1], line[:, 2], 'r-', lw=2, label='muon track')
ax.scatter(*v, c='red', marker='*', s=140, label='vertex')
ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]'); ax.set_zlabel('z [m]')
ax.set_title(f'event {EVENT}: E = {float(ev["energy"]):.2g} GeV')
ax.legend(fontsize=8)

# --- expected light vs perpendicular distance ---
ax2 = fig.add_subplot(122)
ax2.scatter(dp, np.maximum(ly, 1e-8), s=14, c=dl, cmap='viridis')
ax2.set_xscale('log'); ax2.set_yscale('log')
ax2.set_xlabel('perpendicular distance to track [m]')
ax2.set_ylabel('expected photons / OM')
ax2.set_title('light vs distance (colour = distance along track)')
ax2.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Arrival times

`patd_mode=True` returns one dict per optical module in `LightSabrePATD`'s format:
`hit_times`, `residual_times`, `geometric_times`, `num_photons`, `expected_photons`,
`emission_points`, `d_geom`, `t_geom_min`, `vertex_times`, `patd_probs`.

In [ ]:
patd = nusmoothie.light_yield_surrogate(
    event_params=ev, opt_point=geom_dict['points_3d'], patd_mode=True)
if isinstance(patd, dict):
    patd = [patd]

lit = [(i, d) for i, d in enumerate(patd) if d['num_photons'] > 0]
n_ph = sum(d['num_photons'] for _, d in lit)
print(f'{len(lit)} OMs fired, {n_ph} photons in total')

if n_ph:
    t_hit = torch.cat([d['hit_times'] for _, d in lit]).detach().cpu().numpy()
    t_res = torch.cat([d['residual_times'] for _, d in lit]).detach().cpu().numpy()
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].hist(t_hit, bins=40, color='C0')
    ax[0].set_xlabel('hit time [ns]'); ax[0].set_ylabel('photons')
    ax[0].set_title('absolute arrival times'); ax[0].grid(alpha=.3)
    ax[1].hist(t_res, bins=40, color='C1')
    ax[1].axvline(0, color='k', ls=':', lw=1)
    ax[1].set_xlabel('time residual t_hit - t_geom [ns]')
    ax[1].set_title('residuals (negative = PMT jitter)'); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()
    print(f'  t_res: p16/p50/p84 = {np.percentile(t_res, [16, 50, 84]).round(2)} ns, '
          f'frac < 0 = {np.mean(t_res < 0):.3f}')
else:
    print('nothing fired -- move the event closer to the array, raise the energy, '
          'or make the geometry denser')

## Fisher information

Per PMT the three likelihoods contribute independent Fisher matrices that add:

```
F_j = pi (1-pi) grad l grad l^T           (hit, closed form)
    + pi      E_q[ grad log p_q ... ]     (light yield)
    + pi qbar E_t[ grad log p_t ... ]     (arrival time)
```

summed over PMTs, then over strings. `mode='atime'` keeps only the third term but
still evaluates `pi` and `qbar`, since they weight it.

Memory is the thing to watch: the differentiable log-prob holds `FISHER_N_STEPS`
forward + double-backward passes per row, about 2 MB/row in float64. Keep
`fisher_info_chunk_size` small enough that `chunk x 2 MB` fits on the card.

In [ ]:
fisher_loss = FlowFisherResolutionLoss(
    hit_model=nusmoothie.hit_model,
    ly_model=nusmoothie.ly_model,
    atime_model=nusmoothie.atime_model,
    device=device,
    fisher_info_params=('energy', 'zenith', 'azimuth'),
    resolution_type='angular',
    mode='all',
    n_quad=FISHER_N_QUAD,
    n_steps=FISHER_N_STEPS,
    print_loss=True,
)

out = fisher_loss(
    geom_dict,
    signal_event_params=[ev],
    fisher_info_chunk_size=FISHER_CHUNK,
    fisher_res_metric='fom',
    verbose=True,
)

F = out['fisher_info_per_string_per_event']          # (n_events, n_strings, 3, 3)
res_deg = np.degrees(out['angular_resolution_per_event'].detach().cpu().numpy())
print(f"\nFisher per string per event: {tuple(F.shape)}")
print(f"angular resolution: {res_deg[0]:.3f} deg")

F_tot = F.sum(1)[0].detach().cpu().numpy()
print('\ntotal Fisher matrix (log10E, zenith, azimuth):')
print(np.array2string(F_tot, precision=3, suppress_small=False))
print('eigenvalues:', np.linalg.eigvalsh(F_tot).round(4))

## All events, and the arrival-time-only mode

In [ ]:
fisher_atime = FlowFisherResolutionLoss(
    hit_model=nusmoothie.hit_model,
    ly_model=nusmoothie.ly_model,
    atime_model=nusmoothie.atime_model,
    device=device, mode='atime',
    n_quad=FISHER_N_QUAD, n_steps=FISHER_N_STEPS,
)

out_all = fisher_loss(geom_dict, signal_event_params=events,
                      fisher_info_chunk_size=FISHER_CHUNK)
out_at  = fisher_atime(geom_dict, signal_event_params=events,
                       fisher_info_chunk_size=FISHER_CHUNK)

r_all = np.degrees(out_all['angular_resolution_per_event'].detach().cpu().numpy())
r_at  = np.degrees(out_at['angular_resolution_per_event'].detach().cpu().numpy())

print(f"{'ev':>3} {'E [GeV]':>10} {'min d_perp':>11} {'all [deg]':>11} "
      f"{'atime [deg]':>12} {'atime/all':>10}")
for i, ev_i in enumerate(events):
    dpi, _, _ = track_frame(ev_i, pts)
    print(f"{i:>3} {float(ev_i['energy']):>10.3g} {dpi.min():>11.1f} "
          f"{r_all[i]:>11.3f} {r_at[i]:>12.3f} {r_at[i]/r_all[i]:>10.3f}")
print(f"\ncombined (fom): all = {float(out_all['angular_resolution_loss']):.4g} rad, "
      f"atime = {float(out_at['angular_resolution_loss']):.4g} rad")

## Which strings carry the information

`fisher_info_per_string_per_event` is the per-string breakdown the loss sums over, so
`det(F_s)^(1/3)` is a convenient scalar for how much a single string contributes.

In [ ]:
Fs = out_all['fisher_info_per_string_per_event'][0].detach().cpu().numpy()  # (n_strings,3,3)
contrib = np.array([np.linalg.det(m) ** (1 / 3) if np.linalg.det(m) > 0 else 0.0
                    for m in Fs])

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
sc = ax[0].scatter(sxy[:, 0], sxy[:, 1], s=60 + 300 * contrib / max(contrib.max(), 1e-12),
                   c=contrib, cmap='plasma')
v2 = np.asarray(ev['position'], dtype=float).reshape(3)
tl = v2[None, :2] + np.linspace(-400, 800, 2)[:, None] * travel[None, :2]
ax[0].plot(tl[:, 0], tl[:, 1], 'r-', lw=2, label='track (xy)')
ax[0].set_xlabel('x [m]'); ax[0].set_ylabel('y [m]'); ax[0].set_aspect('equal')
ax[0].set_title('per-string Fisher contribution'); ax[0].legend(fontsize=8)
fig.colorbar(sc, ax=ax[0], label='det(F_s)^(1/3)')

order = np.argsort(contrib)[::-1]
ax[1].bar(np.arange(len(contrib)), contrib[order], color='C0')
ax[1].set_xlabel('string (sorted)'); ax[1].set_ylabel('det(F_s)^(1/3)')
ax[1].set_title('contribution ranking'); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f'top string carries {contrib.max()/max(contrib.sum(),1e-12):.1%} of the total')